In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

# pandas setting
pd.set_option('display.max_columns', 500)

# global figure parameters
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.figsize': (8, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

%matplotlib inline


In [ ]:
# claims
filepath='/Volumes/Hammurabi/data/Utah/Utah_files/fact_services_2021.txt'
fact_services_2021=pd.read_csv(filepath,sep='|',nrows=100000,usecols=['MEMBER_KEY_HASHED','PAID_DATE_YEAR','AMT_PAID','DRG_CODE','ICD_DIAG_01'])
fact_services_2021

In [ ]:
# individual level data 
fact_services_2021_1=fact_services_2021.groupby(['MEMBER_KEY_HASHED']).agg({'PAID_DATE_YEAR':'mean','AMT_PAID':'sum','DRG_CODE':'first','ICD_DIAG_01':'first'})
fact_services_2021_1=fact_services_2021_1.reset_index()
fact_services_2021_1

In [ ]:
fact_services_2021_1['AMT_PAID'].hist()
plt.show()

In [ ]:
# demographics
filepath='/Volumes/Hammurabi/data/Utah/Utah_files/dim_member.txt'
dim_member=pd.read_csv(filepath,sep='|',usecols=['MEMBER_KEY_HASHED','MEM_GENDER','MEM_DOB_YEAR','MEM_START_YEAR','MEM_END_YEAR','MEM_ZIP'])
dim_member

In [ ]:
# merge individual level spAmountPaid + demographics
# merge fact_services_2021_1 + dim_member
fact_services_2021_2=pd.merge(fact_services_2021_1,dim_member,how='left',on='MEMBER_KEY_HASHED')
fact_services_2021_2

In [ ]:
# ZIP to 3 digit zip code
fact_services_2021_3=fact_services_2021_2
fact_services_2021_3['MEM_ZIP'].replace('', np.nan, inplace=True)
fact_services_2021_3['Zip3']=fact_services_2021_3['MEM_ZIP'].astype(str).str[:3]
fact_services_2021_3=fact_services_2021_3.drop(columns={'MEM_ZIP'})
fact_services_2021_3['Zip3']=pd.to_numeric(fact_services_2021_3['Zip3'],errors='coerce') # if unconvertible, NaN
fact_services_2021_3

In [ ]:
# load income tax by Zip
filename='/Users/kyun/Downloads/Hammurabi/Wisconsin/income_tax1.csv'
income_tax1=pd.read_csv(filename)
income_tax1=income_tax1[['Zip5','income']]
# convert Zip5 to Zip3
for i in range(len(income_tax1)):
    if len(str(income_tax1['Zip5'].iloc[i]))==4: # if 4 digit zip code
        income_tax1['Zip5'].iloc[i]=int(str(income_tax1['Zip5'].iloc[i])[:2])
    else:  # if 5 digit zip code
        income_tax1['Zip5'].iloc[i]=int(str(income_tax1['Zip5'].iloc[i])[:3])
income_tax1=income_tax1.rename(columns={'Zip5':'Zip3'})
# groupby income average
income_tax1=income_tax1.groupby(['Zip3']).mean()
income_tax1=income_tax1.reset_index()
income_tax1

In [ ]:
# incorporate income_tax1 into fact_services_2021_3
fact_services_2021_4=pd.merge(fact_services_2021_3,income_tax1,on='Zip3',how='left')
fact_services_2021_4

In [ ]:
# Zip code related external data 
MERGE_ALL_ZIP3=pd.read_csv("/Users/kyun/Downloads/Hammurabi/merge20220530/MERGE_ALL_ZIP3_Zia20220530.csv",sep=';')
# remove duplicates
MERGE_ALL_ZIP3_1=MERGE_ALL_ZIP3.drop_duplicates(['Zip'])
MERGE_ALL_ZIP3_1=MERGE_ALL_ZIP3_1.rename(columns={'Zip':'Zip3'})

# convert it to income
MERGE_ALL_ZIP3_2=pd.merge(MERGE_ALL_ZIP3_1,income_tax1,on='Zip3',how='left')
MERGE_ALL_ZIP3_2.drop(columns=['Zip3'],inplace=True)
MERGE_ALL_ZIP3_2

In [ ]:
# fact_services_2021_4 + MERGE_ALL_ZIP3_2, using closest income
fact_services_2021_5=fact_services_2021_4.dropna(subset=['income'])
fact_services_2021_5=fact_services_2021_5.sort_values(by='income')
MERGE_ALL_ZIP3_2=MERGE_ALL_ZIP3_2.dropna(subset=['income'])
MERGE_ALL_ZIP3_2=MERGE_ALL_ZIP3_2.sort_values(by='income')
fact_services_2021_5=pd.merge_asof(fact_services_2021_5, MERGE_ALL_ZIP3_2, on='income', direction='nearest')
fact_services_2021_5

In [ ]:
fact_services_2021_6=pd.get_dummies(fact_services_2021_5, columns=['MEM_GENDER'])  # Gender one hot encoding
fact_services_2021_6

In [ ]:
# fact_services_2021_6 + ICD related data
## fact_services_2021_6 ICD_DIAG_01 -> numeric
ICD10_numeric=pd.read_csv('../Zia_data/ICD10_1_Numeric.txt')
fact_services_2021_7=fact_services_2021_6.rename(columns={'ICD_DIAG_01':'ICD10'})
fact_services_2021_7=pd.merge(fact_services_2021_7, ICD10_numeric,how='left',on='ICD10')
selected_df_total=pd.read_csv('../Zia_data/df_total_selected3.csv')  #ICD10_Zip3.csv # unique ICD10 code and external data
fact_services_2021_7=pd.merge(fact_services_2021_7,selected_df_total,how='left',on='ICD10')
fact_services_2021_7

## ML on fact_services_2021_7

## neural net model

In [ ]:
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense

# select features 
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

data=fact_services_2021_7.apply(lambda x: pd.to_numeric(x, errors='coerce')).fillna(0)

# Load data
X = data.drop(columns=['AMT_PAID']).values
y = data['AMT_PAID'].values

# Feature selection
selector = SelectKBest(f_regression, k=100)
X_new = selector.fit_transform(X, y)

print(X_new)


# split the data into input (X) and output (y) variables
X = X_new

# create a neural network model
model = Sequential()
model.add(Dense(100, input_dim=X_new.shape[1]))
model.add(Dense(100))
model.add(Dense(100))
model.add(Dense(1))

# compile the model
model.compile(loss='mean_squared_error', optimizer='adam')

# fit the model to the data
model.fit(X, y, epochs=100)

# use the model to make predictions
predictions = model.predict(X)

spearman = spearmanr(y, predictions)
pearson = pearsonr(y, predictions)

print(f'Test data Spearman correlation: {spearman[0]:.3}')
print(f'Test data Pearson correlation: {pearson[0][0]:.3}')

# plot SpAmountPaid
plt.plot(y, predictions,'ok')
plt.xlabel("Ground Truth SpAmountPaid")
plt.ylabel("Predicted")

In [ ]:
print(f'Test data Pearson correlation: {pearson[0][0]:.3}')

# plot SpAmountPaid
plt.plot(y, predictions,'ok')
plt.xlabel("Ground Truth SpAmountPaid")
plt.ylabel("Predicted")